### 1. Baseline Logistic Regression on Adult dataset
- No EDA
- No feature engineering, imputing, etc.
- No privacy
- Model is one-hot encoding of cat vars, rescaling integer vars, and vanilla Logistic Regression


Problem: classify whether less than or greater than $50K

In [5]:
# Read from file and build pandas DataFrame

import pandas as pd
import re

# import column names from adult.names
with open('../data/adult/adult.names') as fp:
    cols = []
    for line in fp:
        sre = re.match(r'(?P<colname>[a-z\-]+):.*\.', line)
        if sre:
            cols.append(sre.group('colname'))
    cols.append('label')

options = {'header': None, 'names': cols, 'skipinitialspace': True}

# adult.data
train_df = pd.read_csv('../data/adult/adult.data', **options)

# adult.test
test_df = pd.read_csv('../data/adult/adult.test', skiprows=1, **options)
test_df['label'] = test_df['label'].str.rstrip('.')


In [6]:
print(train_df.info())
train_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  label           32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
None


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,label
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [7]:
test_df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,label
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16276,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K
16277,64,?,321403,HS-grad,9,Widowed,?,Other-relative,Black,Male,0,0,40,United-States,<=50K
16278,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K
16279,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K


In [8]:
# some preprocessing

X_train = train_df.iloc[:,:-1]
y_train = train_df[['label']].rename(columns={'label': 'income'})

X_test = train_df.iloc[:,:-1]
y_test = train_df[['label']].rename(columns={'label': 'income'})

In [9]:
X_train

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States


In [13]:
y_train['income'] = y_train['income'].apply(lambda x: 0 if x[0] == '<' else 1)
y_test['income'] = y_test['income'].apply(lambda x: 0 if x[0] == '<' else 1)
y_test

,income
0,0
1,0
2,0
3,0
4,0
...,...
32556,0
32557,1
32558,0
32559,0


In [ ]:
# get different var types
integer_vars = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_vars = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country']
binary_vars = ['sex'] # don't include income

In [ ]:
# do vanilla Logistic Regression

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Create the preprocessing stages
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), integer_vars),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_vars),
        ("bin", OneHotEncoder(drop="if_binary"), binary_vars),
    ],
    remainder="passthrough",  # This ensures target 'income' is kept unchanged
)

# 2. Construct the  pipeline
vanilla_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(solver="lbfgs")),
    ]
)

vanilla_pipeline.fit(X_train, y_train)

# 3. Make predictions
predictions = vanilla_pipeline.predict(X_test)

/Users/ammareltigani/miniconda3/envs/erdos_project_environment/lib/python3.12/site-packages/sklearn/utils/validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [17]:
from sklearn.metrics import accuracy_score, precision_score

# 4. Get accuracy and precision

accuracy = accuracy_score(y_test, predictions)
# Precision: Out of all positive predictions, how many were actually positive?
precision = precision_score(y_test, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")

Accuracy:  0.8530
Precision: 0.7380


### 2. Differentially Private Logistic Regression (using IBM's Diffprivlib)
https://github.com/IBM/differential-privacy-library/blob/main/notebooks/logistic_regression.ipynb

**Note**: The underlying mechanism that ensures differential privacy (class `Vector`) does not directly add noise to the entries of the data but rather modified the convex objective function of logistic regression loss. This means we cannot directly feed any dataset from here to our deanonymizer, but running this serves instead as an auxiliary downstream task benchmark to our own differentially private solution.


In [ ]:
# Install Install package. MAKE SURE correct conda env is activated before!
%pip install diffprivlib

In [ ]:
#TODO: looks like there is some versioning issues.. Will skip this now and come back later..
import diffprivlib.models as dp



ImportError: cannot import name 'DOUBLE' from 'sklearn.tree._tree' (/Users/ammareltigani/miniconda3/envs/erdos_project_environment/lib/python3.12/site-packages/sklearn/tree/_tree.cpython-312-darwin.so)

$\color{red}{\text{TODO: Skip steps (3)-(6) for now as they are much more time-consuming. Come back to this later...}}$

### 3. Logistic Regression with (custom) random Gaussian noise added to numerical variables

### 4. Logistic Regression with (custom) categorical variables changed to a random different class with probability $p$

**Note**: The choices of features to inject noise into and/or change should be based on the QIs. To make this more effective and work towards our goal of adversarial improvement, we could also do this for other non-QIs features that have "large coefficients" in the de-anonymization step.

### 5. Logistic Regression with $k$-anonymity, $\ell$-diversity, and $t$-closeness (using AnonyPy and pyCANON)

- For non-numerical (categorical) data, want to do Semantic Resampling: https://link.springer.com/chapter/10.1007/978-3-642-31724-8_54
- AnonyPy: https://pypi.org/project/anonypy\
- pyCANON: https://pycanon.readthedocs.io/en/latest/

### 6. Combining both (2) and (3), (2) and (4), and (3) and (4)

### 7. Logistic Regression on synthetic dataset (using SDV SmartNoise-Synth (OpenDP) library)
- SDV: https://docs.sdv.dev/sdv/single-table-data/modeling/synthesizers
- SmartNoise-Synth: https://github.com/TeDiou/smartnoise-sdk



In [2]:
# Let's go with SVD
%pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 14.4 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [sdv]32m 9/10 [sdv]core]
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata

X_train = pd.read_csv('../data/adult/adult_with_pii.csv')

metadata = Metadata.detect_from_dataframe(X_train)

synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(X_train)
synthetic_data = synthesizer.sample(num_rows=len(X_train))

/Users/ammareltigani/miniconda3/envs/erdos_project_environment/lib/python3.12/site-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [18]:
metadata

{
    "tables": {
        "table": {
            "primary_key": "Name",
            "columns": {
                "Name": {
                    "sdtype": "id"
                },
                "DOB": {
                    "sdtype": "categorical"
                },
                "SSN": {
                    "pii": true,
                    "sdtype": "ssn"
                },
                "Zip": {
                    "sdtype": "numerical"
                },
                "Age": {
                    "sdtype": "numerical"
                },
                "Workclass": {
                    "sdtype": "categorical"
                },
                "fnlwgt": {
                    "sdtype": "numerical"
                },
                "Education": {
                    "sdtype": "categorical"
                },
                "Education-Num": {
                    "sdtype": "numerical"
                },
                "Martial Status": {
                    "sdtype": "categorica

In [19]:
X_train

,Name,DOB,SSN,Zip,Age,Workclass,fnlwgt,Education,Education-Num,Martial Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,Target
0,Karrie Trusslove,9/7/67,732-14-6110,64152,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,Brandise Tripony,6/7/88,150-19-2766,61523,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,Brenn McNeely,8/6/91,725-59-9860,95668,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,Dorry Poter,4/6/09,659-57-4974,25503,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,Dick Honnan,9/16/51,220-93-3811,75387,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,Ardyce Golby,10/29/61,212-61-8338,41328,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,Jean O'Connor,6/28/52,737-32-2919,94735,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,Reuben Skrzynski,8/9/66,314-48-0219,49628,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,Caye Biddle,5/19/78,647-75-3550,8213,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [20]:
synthetic_data

,Name,DOB,SSN,Zip,Age,Workclass,fnlwgt,Education,Education-Num,Martial Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,Target
0,sdv-id-tMHSIB,9/12/55,141-41-5107,67885,23,Private,130869,Bachelors,11,Married-civ-spouse,Craft-repair,Unmarried,White,Male,243,1,33,United-States,<=50K
1,sdv-id-XKskQq,6/14/70,658-19-6189,59563,29,Private,137670,Some-college,8,Married-civ-spouse,Prof-specialty,Unmarried,White,Male,6743,1022,22,United-States,>50K
2,sdv-id-gGNAFl,3/25/85,868-79-2363,42104,20,State-gov,101406,7th-8th,8,Never-married,Other-service,Own-child,White,Male,10611,122,32,United-States,<=50K
3,sdv-id-FXLpaC,10/28/98,185-04-6553,71920,42,Private,139052,HS-grad,11,Married-civ-spouse,Transport-moving,Wife,Other,Male,16356,0,20,United-States,<=50K
4,sdv-id-VshhmF,7/8/07,164-69-5320,53752,50,Private,245408,Bachelors,8,Never-married,Sales,Wife,White,Male,52578,64,27,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,sdv-id-tJlMJG,3/17/55,539-33-2219,97590,57,Private,225981,Doctorate,9,Married-civ-spouse,Exec-managerial,Husband,White,Female,12012,32,35,United-States,>50K
32557,sdv-id-YckinC,2/20/91,883-36-9314,61236,26,Private,39347,HS-grad,11,Never-married,Sales,Not-in-family,White,Male,9229,0,26,United-States,<=50K
32558,sdv-id-LTwtWH,8/30/63,889-63-5465,40106,25,State-gov,208555,Bachelors,8,Married-civ-spouse,Other-service,Not-in-family,White,Male,44737,0,42,United-States,<=50K
32559,sdv-id-VIQzGe,5/13/85,841-46-9396,17571,42,Private,360913,HS-grad,11,Never-married,Machine-op-inspct,Not-in-family,White,Male,62327,299,68,United-States,<=50K


In [21]:
# Should remove redundant columns: (dob/age, education/education_no, marital-status/relationship)
X_train[(X_train['Age'] == 39) & (X_train['Education'] == 'Bachelors')]

,Name,DOB,SSN,Zip,Age,Workclass,fnlwgt,Education,Education-Num,Martial Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,Target
0,Karrie Trusslove,9/7/67,732-14-6110,64152,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
848,Geoff Bruckman,8/26/54,573-19-1166,30123,39,Private,138192,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K
915,Holmes Shermar,3/26/05,108-49-6435,28145,39,Private,202027,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,15024,0,45,United-States,>50K
999,Marcelia Hunnam,11/22/08,730-47-9143,95637,39,Self-emp-inc,329980,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,15024,0,50,United-States,>50K
1336,Jamesy Matthewman,2/16/92,171-92-4606,10184,39,Private,174938,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Male,0,0,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31491,Murielle Lyste,11/19/01,373-74-4232,290,39,Private,139012,Bachelors,13,Married-civ-spouse,Adm-clerical,Husband,Asian-Pac-Islander,Male,0,0,40,Philippines,>50K
31868,Elijah Blasius,10/3/51,444-86-6914,94982,39,Private,75891,Bachelors,13,Divorced,Tech-support,Unmarried,White,Female,0,0,40,United-States,<=50K
31872,Charin Velareal,11/13/77,735-56-7574,44742,39,Private,174242,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,15024,0,60,United-States,>50K
32092,Sherie Emson,2/9/86,841-81-5398,95007,39,Private,30529,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,1887,40,United-States,>50K


In [22]:
synthetic_data[(synthetic_data['Age'] == 39) & (synthetic_data['Education'] == 'Bachelors')]

,Name,DOB,SSN,Zip,Age,Workclass,fnlwgt,Education,Education-Num,Martial Status,Occupation,Relationship,Race,Sex,Capital Gain,Capital Loss,Hours per week,Country,Target
11,sdv-id-GaETiD,12/3/73,398-02-0861,15904,39,Self-emp-not-inc,141291,Bachelors,9,Separated,Protective-serv,Husband,White,Male,2611,516,54,Mexico,<=50K
576,sdv-id-vyVAnB,2/7/60,111-73-4447,85897,39,Private,187844,Bachelors,14,Divorced,Handlers-cleaners,Unmarried,White,Male,60342,13,32,United-States,<=50K
1251,sdv-id-wfRxQW,6/11/86,632-25-2735,57855,39,Private,335868,Bachelors,12,Never-married,Craft-repair,Not-in-family,White,Female,32345,6,36,United-States,<=50K
1727,sdv-id-quYaAl,12/26/83,229-47-6023,68224,39,Private,148931,Bachelors,11,Never-married,NaN,Husband,White,Male,5203,1,38,Mexico,<=50K
1786,sdv-id-IRXcVb,1/25/96,691-81-0406,25142,39,Private,65257,Bachelors,15,Never-married,Other-service,Not-in-family,White,Female,19628,23,66,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31876,sdv-id-HBOvsC,12/17/09,763-35-8937,75318,39,Private,319338,Bachelors,14,Married-civ-spouse,Prof-specialty,Husband,White,Female,6469,0,56,NaN,<=50K
31913,sdv-id-uqJdFS,10/14/84,882-15-3021,69913,39,Private,94975,Bachelors,10,Married-civ-spouse,Adm-clerical,Wife,White,Male,9836,178,38,NaN,>50K
31919,sdv-id-YoJZGJ,1/5/86,782-76-0647,4561,39,Private,344431,Bachelors,11,Married-civ-spouse,NaN,Unmarried,Black,Female,712,92,46,United-States,<=50K
32253,sdv-id-qquwVQ,12/14/61,444-32-1879,19608,39,Private,87667,Bachelors,4,Married-civ-spouse,Other-service,Unmarried,White,Male,3723,0,62,China,>50K


As a first step, we can run a **diagnostic** to ensure that the data is valid. SDV's diagnostic performs some basic checks such as:

- All primary keys must be unique
- Continuous values must adhere to the min/max of the real data
- Discrete columns (non-PII) must have the same categories as the real data
- Etc.


In [23]:
from sdv.evaluation.single_table import run_diagnostic

diagnostic = run_diagnostic(
    real_data=X_train,
    synthetic_data=synthetic_data,
    metadata=metadata
)

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 19/19 [00:00<00:00, 332.87it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 637.34it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



We can also measure the **data quality** or the statistical similarity between the real and synthetic data. This value may vary anywhere from 0 to 100%.

In [24]:
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    X_train,
    synthetic_data,
    metadata
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 19/19 [00:01<00:00, 12.44it/s]|
Column Shapes Score: 87.48%

(2/2) Evaluating Column Pair Trends: |██████████| 171/171 [00:01<00:00, 151.76it/s]|
Column Pair Trends Score: 56.09%

Overall Score (Average): 71.78%



In [25]:
quality_report.get_details('Column Shapes')

,Column,Metric,Score
0,DOB,TVComplement,0.732195
1,Zip,KSComplement,0.995117
2,Age,KSComplement,0.977826
3,Workclass,TVComplement,0.995369
4,fnlwgt,KSComplement,0.960689
5,Education,TVComplement,0.993182
6,Education-Num,KSComplement,0.861399
7,Martial Status,TVComplement,0.991094
8,Occupation,TVComplement,0.991991
9,Relationship,TVComplement,0.995485


In [26]:
from sdv.evaluation.single_table import get_column_plot

fig = get_column_plot(
    real_data=X_train,
    synthetic_data=synthetic_data,
    column_name='hours-per-week',
    metadata=metadata
)

fig.show()

TypeError: 'NoneType' object is not subscriptable

In [15]:
X_train_s = synthetic_data.drop(columns=['Name','Target', 'DOB', 'SSN', 'Zip'])
y_train_s = synthetic_data[['Target']]